# TP 1 · Préparer les données pour un modèle

AI4I 2020. On transforme le dataset brut en matrices propres pour scikit-learn.

**Objectif :**
1. Construire X avec les six features autorisées et isoler Y
2. Créer des splits stratifiés 60/20/20, `random_state=42`
3. Préparer les colonnes numériques et la catégorie Type
4. Vérifier les dimensions et expliquer chaque `fit`

## 0. Setup

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

TARGET = "Machine failure"
FEATURES = ["Torque [Nm]", "Type", "Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]", "Tool wear [min]"]
df = pd.read_csv("data/ai4i2020.csv")

## 1. Construire X et Y

- **X** = les entrées du modèle → un **DataFrame** (2D : lignes × 6 colonnes)
- **Y** = la réponse à prédire → une **Series** (1D : une valeur par ligne)

In [2]:
X = df[FEATURES]
Y = df[TARGET]

In [3]:
print(X.shape, Y.shape)
print(type(X), type(Y))

(10000, 6) (10000,)
<class 'pandas.DataFrame'> <class 'pandas.Series'>


## 2. Splits 60/20/20

`train_test_split` ne coupe qu'en **deux**. Pour avoir trois morceaux, on l'appelle deux fois :

```
10 000 ──(test_size=0.4)──► train 6000 (60 %)
                       └──► tmp   4000 ──(test_size=0.5)──► val  2000 (20 %)
                                                      └──► test 2000 (20 %)
```

| Split | Rôle |
|---|---|
| **train** | le modèle apprend dessus |
| **val** | on compare les modèles / réglages dessus |
| **test** | évaluation finale, **une seule fois** à la fin |

- `stratify=Y` : garde le même % de pannes dans chaque split. Sans ça, avec 3,4 % de pannes, un split pourrait en avoir très peu par hasard.
- `random_state=42` : le mélange est le même à chaque exécution → résultats reproductibles.

In [4]:
X_train, X_tmp, Y_train, Y_tmp = train_test_split(X, Y, test_size=0.4, stratify=Y, random_state=42)

In [5]:
X_val, X_test, Y_val, Y_test = train_test_split(X_tmp, Y_tmp, test_size=0.5, stratify=Y_tmp, random_state=42)

In [6]:
print(len(X_train), len(X_val), len(X_test))
print(Y_train.mean(), Y_val.mean(), Y_test.mean())


6000 2000 2000
0.03383333333333333 0.034 0.034


**Lecture :** 6000 / 2000 / 2000 ✅, et ~3,4 % de pannes dans les trois splits ✅ → la stratification a marché.

In [7]:
X_train.describe()

,Torque [Nm],Air temperature [K],Process temperature [K],Rotational speed [rpm],Tool wear [min]
count,6000.000000,6000.000000,6000.000000,6000.000000,6000.000000
mean,40.034067,300.019783,310.012383,1538.869500,107.492833
std,10.035769,2.004329,1.479536,180.530086,63.571628
min,3.800000,295.300000,305.700000,1168.000000,0.000000
25%,33.200000,298.300000,308.800000,1422.000000,52.000000
50%,40.100000,300.100000,310.100000,1503.000000,107.000000
75%,46.900000,301.500000,311.100000,1614.000000,162.000000
max,76.600000,304.500000,313.800000,2886.000000,253.000000


## 3. Preprocessing

Les modèles ne comprennent que des nombres, et sont sensibles aux échelles :
- **Colonnes numériques** → `StandardScaler` : `(x − moyenne) / écart-type`. Sans ça, Rotational speed (~1500) écraserait Process temperature (écart-type ~1,5).
- **Type** (texte L/M/H) → `OneHotEncoder` : une colonne 0/1 par catégorie (`Type_H`, `Type_L`, `Type_M`).

`ColumnTransformer` applique le bon traitement à chaque groupe de colonnes.

In [8]:
NUM_COLS = ["Torque [Nm]", "Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]", "Tool wear [min]"]
CAT_COLS = ["Type"]

In [9]:
preprocess = ColumnTransformer([("num", StandardScaler(), NUM_COLS), ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS)])

### `fit` vs `transform` : la règle d'or

- `fit` = **apprendre** les paramètres (moyennes, écarts-types, catégories)
- `transform` = **appliquer** ces paramètres

On fait `fit` **uniquement sur train**, puis `transform` sur val et test. Si on faisait `fit` sur val/test, le preprocessing « verrait » des données censées être inconnues → **fuite d'information**, et des scores trop optimistes.

In [10]:
X_train_p = preprocess.fit_transform(X_train)
X_val_p = preprocess.transform(X_val)
X_test_p = preprocess.transform(X_test)


## 4. Vérifications

In [11]:
print(X_train_p.shape, X_val_p.shape, X_test_p.shape)
print(preprocess.get_feature_names_out())

(6000, 8) (2000, 8) (2000, 8)
['num__Torque [Nm]' 'num__Air temperature [K]'
 'num__Process temperature [K]' 'num__Rotational speed [rpm]'
 'num__Tool wear [min]' 'cat__Type_H' 'cat__Type_L' 'cat__Type_M']


**Lecture :** 6 colonnes → **8** : les 5 numériques restent 5, et Type devient 3 colonnes (H, L, M). Le nombre de lignes ne change pas.

In [12]:
scaler = preprocess.named_transformers_["num"]
print(scaler.mean_)
print(scaler.scale_)

[  40.03406667  300.01978333  310.01238333 1538.8695      107.49283333]
[ 10.03493246   2.00416241   1.4794126  180.5150413   63.56632978]


In [13]:
encoder = preprocess.named_transformers_["cat"]
print(encoder.categories_)

[array(['H', 'L', 'M'], dtype=object)]


**Lecture :** ce sont les paramètres appris pendant le `fit` — **calculés sur train uniquement**. Par exemple torque : moyenne 40,0, écart-type 10,0 (identiques au `describe()` de train plus haut). L'encoder a retenu les 3 catégories H, L, M.

In [14]:
print(X_train_p[:, :5].mean(axis=0).round(3))
print(X_val_p[:, :5].mean(axis=0).round(3))
print(X_test_p[:, :5].mean(axis=0).round(3))

[ 0. -0.  0. -0. -0.]
[-0.012 -0.029 -0.024 -0.002  0.004]
[-0.011 -0.008  0.001 -0.001  0.032]


**Lecture :**
- **train** : moyennes exactement 0 → normal, le scaler a été calculé sur ces données.
- **val / test** : moyennes *proches* de 0 mais pas exactement (−0,03 à 0,03) → normal aussi, on leur applique les paramètres de train. Si c'était exactement 0, ça voudrait dire qu'on a fait un `fit` dessus (erreur).

## Bilan

`X_train_p`, `X_val_p`, `X_test_p` sont prêts pour un modèle. C'est exactement ce setup qu'on reprend au début des TP 3 à 6.